# Chapter 13: AI Project ROI Calculator

**From Enterprise Architect to AI Enterprise Architect**

Build an AI project ROI calculator that models the business case for AI implementation.
Input your use case parameters — volume, current cost, error rate — and compare
conservative, moderate, and aggressive adoption scenarios.

**Key Insight:** Framing AI as augmentation with clear ROI builds executive support.
Numbers speak louder than demos in the boardroom.

---

## Setup

This notebook is self-contained and does not require any external APIs.
It uses only Python standard library and tabulate for display.

In [ ]:
# Install dependencies
!pip install -q tabulate

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List
from tabulate import tabulate
import json

## Define the ROI Calculator

The `AIROICalculator` class encapsulates all the inputs for a single AI use case
and computes financial outcomes across three adoption scenarios:

| Scenario | Automation Rate | Description |
|---|---|---|
| Conservative | 30% | Pilot phase — only straightforward cases automated |
| Moderate | 60% | Scaled rollout — most standard cases automated |
| Aggressive | 85% | Full production — only edge cases remain manual |

For each scenario we compute:
- **Monthly savings** from reduced per-task cost on automated volume
- **Error reduction** (percentage improvement)
- **Time saved** in hours per month
- **ROI months** — payback period for the implementation investment
- **Annual net benefit** after accounting for platform costs

In [ ]:
@dataclass
class AIROICalculator:
    """Calculate ROI for an AI implementation across adoption scenarios."""

    # Use case identification
    process_name: str

    # Current state (manual / legacy process)
    volume_per_month: int              # Number of tasks per month
    current_cost_per_task: float       # Cost in dollars per task
    current_error_rate: float          # Error rate as a fraction (0-1)
    current_time_per_task_minutes: float  # Minutes per task

    # Projected AI state
    ai_cost_per_task: float            # Cost in dollars per task with AI
    ai_error_rate: float               # Projected AI error rate (0-1)
    ai_time_per_task_minutes: float    # Minutes per task with AI

    # Investment
    implementation_cost: float         # One-time implementation cost
    monthly_platform_cost: float       # Ongoing monthly platform cost

    # Adoption scenarios: name -> fraction of volume automated
    SCENARIOS: Dict[str, float] = field(default_factory=lambda: {
        "Conservative (30%)": 0.30,
        "Moderate (60%)": 0.60,
        "Aggressive (85%)": 0.85,
    })

    def calculate_scenario(self, adoption_rate: float) -> Dict:
        """Calculate financials for a single adoption scenario."""
        automated_volume = self.volume_per_month * adoption_rate
        manual_volume = self.volume_per_month - automated_volume

        # Current total monthly cost (everything is manual)
        current_monthly_cost = self.volume_per_month * self.current_cost_per_task

        # New monthly cost: automated portion at AI rate + manual remainder
        new_monthly_cost = (
            automated_volume * self.ai_cost_per_task
            + manual_volume * self.current_cost_per_task
            + self.monthly_platform_cost
        )

        monthly_savings = current_monthly_cost - new_monthly_cost

        # Error reduction: weighted blend of AI and manual error rates
        blended_error_rate = (
            (automated_volume * self.ai_error_rate
             + manual_volume * self.current_error_rate)
            / self.volume_per_month
        )
        error_reduction_pct = (
            (self.current_error_rate - blended_error_rate)
            / self.current_error_rate * 100
            if self.current_error_rate > 0 else 0
        )

        # Time saved
        current_total_minutes = self.volume_per_month * self.current_time_per_task_minutes
        new_total_minutes = (
            automated_volume * self.ai_time_per_task_minutes
            + manual_volume * self.current_time_per_task_minutes
        )
        time_saved_hours = (current_total_minutes - new_total_minutes) / 60

        # Payback period
        roi_months = (
            self.implementation_cost / monthly_savings
            if monthly_savings > 0 else float("inf")
        )

        # Annual net benefit (year 1 accounts for implementation cost)
        annual_net_benefit = (monthly_savings * 12) - self.implementation_cost

        return {
            "automated_volume": int(automated_volume),
            "monthly_savings": round(monthly_savings, 2),
            "error_reduction_pct": round(error_reduction_pct, 1),
            "time_saved_hours": round(time_saved_hours, 1),
            "roi_months": round(roi_months, 1),
            "annual_net_benefit": round(annual_net_benefit, 2),
        }

    def run_all_scenarios(self) -> Dict[str, Dict]:
        """Run calculations for all adoption scenarios."""
        return {
            name: self.calculate_scenario(rate)
            for name, rate in self.SCENARIOS.items()
        }

    def display_results(self) -> None:
        """Display a formatted comparison table across all scenarios."""
        results = self.run_all_scenarios()

        print(f"\n{'='*70}")
        print(f"  ROI Analysis: {self.process_name}")
        print(f"{'='*70}")
        print(f"  Volume: {self.volume_per_month:,} tasks/month")
        print(f"  Current cost: ${self.current_cost_per_task:.2f}/task  |  "
              f"AI cost: ${self.ai_cost_per_task:.2f}/task")
        print(f"  Current error rate: {self.current_error_rate*100:.1f}%  |  "
              f"AI error rate: {self.ai_error_rate*100:.1f}%")
        print(f"  Implementation cost: ${self.implementation_cost:,.0f}  |  "
              f"Monthly platform: ${self.monthly_platform_cost:,.0f}")
        print(f"{'='*70}\n")

        headers = ["Metric"] + list(results.keys())
        rows = [
            ["Automated Volume"] + [
                f"{r['automated_volume']:,}" for r in results.values()
            ],
            ["Monthly Savings"] + [
                f"${r['monthly_savings']:,.0f}" for r in results.values()
            ],
            ["Error Reduction"] + [
                f"{r['error_reduction_pct']:.1f}%" for r in results.values()
            ],
            ["Time Saved (hrs/mo)"] + [
                f"{r['time_saved_hours']:,.0f}" for r in results.values()
            ],
            ["Payback Period"] + [
                f"{r['roi_months']:.1f} months" for r in results.values()
            ],
            ["Year-1 Net Benefit"] + [
                f"${r['annual_net_benefit']:,.0f}" for r in results.values()
            ],
        ]

        print(tabulate(rows, headers=headers, tablefmt="grid"))
        print()

## Use Case 1: Document Processing

A large enterprise processes 10,000 documents per month — invoices, contracts,
onboarding forms. Manual processing costs $15 per document and takes 20 minutes.
An AI-powered document extraction pipeline brings the cost down to $2 per document
with a 2-minute processing time and a much lower error rate.

In [ ]:
doc_processing = AIROICalculator(
    process_name="Document Processing",
    volume_per_month=10_000,
    current_cost_per_task=15.00,
    current_error_rate=0.05,         # 5% manual error rate
    current_time_per_task_minutes=20,
    ai_cost_per_task=2.00,
    ai_error_rate=0.01,              # 1% AI error rate
    ai_time_per_task_minutes=2,
    implementation_cost=150_000,     # One-time build + integration
    monthly_platform_cost=5_000,     # Cloud infra + API costs
)

doc_processing.display_results()

## Use Case 2: Customer Support Triage

A company handles 50,000 support tickets per month. Human agents cost $8 per
ticket on average (including routing, initial response, and escalation). An AI
triage and response system handles routine tickets for $1.50 each, freeing agents
to focus on complex cases.

In [ ]:
customer_support = AIROICalculator(
    process_name="Customer Support Triage",
    volume_per_month=50_000,
    current_cost_per_task=8.00,
    current_error_rate=0.12,         # 12% misrouting / wrong initial response
    current_time_per_task_minutes=15,
    ai_cost_per_task=1.50,
    ai_error_rate=0.04,              # 4% AI error rate
    ai_time_per_task_minutes=1,
    implementation_cost=200_000,
    monthly_platform_cost=12_000,
)

customer_support.display_results()

## Use Case 3: Compliance Review

Regulatory compliance reviews are expensive — $60 per review with specialized
staff spending 45 minutes each. AI-assisted review reduces cost to $12 per review
by pre-screening documents and highlighting issues, cutting review time to 10 minutes
for the remaining human verification step.

In [ ]:
compliance_review = AIROICalculator(
    process_name="Compliance Review",
    volume_per_month=4_000,
    current_cost_per_task=60.00,
    current_error_rate=0.03,         # 3% — compliance staff are careful
    current_time_per_task_minutes=45,
    ai_cost_per_task=12.00,
    ai_error_rate=0.02,              # 2% — AI is slightly better
    ai_time_per_task_minutes=10,
    implementation_cost=300_000,     # Higher due to regulatory requirements
    monthly_platform_cost=8_000,
)

compliance_review.display_results()

## Cross-Use-Case Comparison

Let's compare all three use cases side by side at the **moderate (60%)** adoption
scenario — this is typically the most realistic target for year one.

In [ ]:
# Compare all three use cases at the moderate scenario
use_cases = [doc_processing, customer_support, compliance_review]

moderate_results = []
for uc in use_cases:
    result = uc.calculate_scenario(0.60)  # 60% adoption
    moderate_results.append(result)

headers = ["Metric"] + [uc.process_name for uc in use_cases]
rows = [
    ["Monthly Savings"] + [f"${r['monthly_savings']:,.0f}" for r in moderate_results],
    ["Error Reduction"] + [f"{r['error_reduction_pct']:.1f}%" for r in moderate_results],
    ["Time Saved (hrs/mo)"] + [f"{r['time_saved_hours']:,.0f}" for r in moderate_results],
    ["Payback (months)"] + [f"{r['roi_months']:.1f}" for r in moderate_results],
    ["Year-1 Net Benefit"] + [f"${r['annual_net_benefit']:,.0f}" for r in moderate_results],
]

print("Cross-Use-Case Comparison (Moderate / 60% Adoption)")
print("=" * 70)
print(tabulate(rows, headers=headers, tablefmt="grid"))

## Sensitivity Analysis: What If AI Error Rate Is 2x Expected?

One of the most common risks in AI projects is that real-world error rates exceed
lab benchmarks. Let's stress-test our models by doubling the projected AI error rate
and recalculating. This helps answer the executive question: "What if the AI isn't
as good as you say?"

Note: higher error rates don't directly change the cost model in our calculator,
but they *do* change the error reduction metric and, in practice, would increase
rework costs. We'll model the rework impact as well.

In [ ]:
def sensitivity_analysis(calculator: AIROICalculator, error_multiplier: float = 2.0):
    """Compare baseline vs. degraded AI error rate scenarios."""
    # Baseline
    baseline = calculator.calculate_scenario(0.60)

    # Degraded: double the AI error rate
    degraded_error_rate = min(calculator.ai_error_rate * error_multiplier, 1.0)

    # Estimate rework cost: each AI error costs 50% of the manual task cost to fix
    rework_cost_per_error = calculator.current_cost_per_task * 0.50
    automated_volume = calculator.volume_per_month * 0.60

    baseline_rework = automated_volume * calculator.ai_error_rate * rework_cost_per_error
    degraded_rework = automated_volume * degraded_error_rate * rework_cost_per_error
    rework_increase = degraded_rework - baseline_rework

    # Adjusted savings
    adjusted_monthly_savings = baseline["monthly_savings"] - rework_increase
    adjusted_roi_months = (
        calculator.implementation_cost / adjusted_monthly_savings
        if adjusted_monthly_savings > 0 else float("inf")
    )
    adjusted_annual = (adjusted_monthly_savings * 12) - calculator.implementation_cost

    print(f"\nSensitivity Analysis: {calculator.process_name}")
    print(f"  Scenario: AI error rate {calculator.ai_error_rate*100:.1f}% → "
          f"{degraded_error_rate*100:.1f}% (2x)")
    print(f"{'='*60}")

    headers = ["Metric", "Baseline", "2x Error Rate", "Impact"]
    rows = [
        [
            "Monthly Rework Cost",
            f"${baseline_rework:,.0f}",
            f"${degraded_rework:,.0f}",
            f"+${rework_increase:,.0f}",
        ],
        [
            "Adj. Monthly Savings",
            f"${baseline['monthly_savings']:,.0f}",
            f"${adjusted_monthly_savings:,.0f}",
            f"-${rework_increase:,.0f}",
        ],
        [
            "Payback Period",
            f"{baseline['roi_months']:.1f} mo",
            f"{adjusted_roi_months:.1f} mo",
            f"+{adjusted_roi_months - baseline['roi_months']:.1f} mo",
        ],
        [
            "Year-1 Net Benefit",
            f"${baseline['annual_net_benefit']:,.0f}",
            f"${adjusted_annual:,.0f}",
            f"-${baseline['annual_net_benefit'] - adjusted_annual:,.0f}",
        ],
    ]

    print(tabulate(rows, headers=headers, tablefmt="grid"))

    # Verdict
    if adjusted_annual > 0:
        print(f"\n  ✓ Still profitable even with 2x error rate.")
    else:
        print(f"\n  ✗ Project becomes unprofitable at 2x error rate — reduce risk first.")


# Run sensitivity analysis for all three use cases
for uc in use_cases:
    sensitivity_analysis(uc)

## Try It Yourself

Plug in your own use case parameters below. Change the numbers to match a real
process in your organization and see how the ROI looks across scenarios.

In [ ]:
# ============================================================
# YOUR USE CASE — edit these values
# ============================================================
my_use_case = AIROICalculator(
    process_name="My AI Project",
    volume_per_month=5_000,
    current_cost_per_task=25.00,
    current_error_rate=0.08,
    current_time_per_task_minutes=30,
    ai_cost_per_task=5.00,
    ai_error_rate=0.02,
    ai_time_per_task_minutes=5,
    implementation_cost=200_000,
    monthly_platform_cost=6_000,
)

my_use_case.display_results()
sensitivity_analysis(my_use_case)

## Key Takeaways

1. **Start with the moderate scenario** when presenting to executives — it's
   credible and still compelling.
2. **Always include sensitivity analysis.** Showing you've stress-tested the
   numbers builds trust.
3. **Payback period matters more than total savings** for getting initial funding.
   A 3-month payback is an easy yes; a 24-month payback needs more justification.
4. **Error reduction is often more valuable than cost savings** in regulated
   industries — quantify it separately.
5. **Time saved translates to capacity** — staff can be redirected to higher-value
   work rather than eliminated.

Use this calculator as a template for your AI business cases. The numbers don't
lie, and they make the architect's recommendation concrete.